# 04 - Convolutional neural networks: learning spatial features

**Goal:** apply a filter by hand, track image shapes through a CNN, train a small network,
and inspect its learned filters and feature maps.

**Before you start:** lessons 02-03's training loop, logits, activations, and validation.
This notebook loads its own dataset and trains from scratch. No previous checkpoint is needed.
Use **Restart Kernel and Run All Cells** to present it from a clean state.

**Live demonstration:** 45-60 minutes including discussion. A small FashionMNIST subset
and compact model are suitable for a local NVIDIA GPU; exact runtime varies. Smoke mode
uses only a few batches and cannot demonstrate model quality.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


## 1. Configure the run

Convolutions reuse a small set of weights across locations. That makes them a useful
architectural assumption for images: a visual pattern can be relevant in several places.
It does not make a network perfectly invariant to shifts, rotation, or scale.


In [ ]:
from torch import nn
import torch.nn.functional as F
from torchvision import transforms

SEED = 42
cu.seed_everything(SEED)
TRAIN_N = 192 if SMOKE else (6000 if QUICK else 48000)
VAL_N = 48 if SMOKE else (800 if QUICK else 12000)
TEST_N = 48 if SMOKE else (1000 if QUICK else 10000)
EPOCHS = 1 if SMOKE else (5 if QUICK else 15)
BATCH_SIZE = 64
print({"device": str(DEVICE), "train_examples": TRAIN_N, "epochs": EPOCHS})


## 2. Load and inspect FashionMNIST

Inputs remain `[N, 1, 28, 28]` through the convolutional layers. We flatten only after
learning spatial features. Fixed normalization maps the grayscale inputs to `[-1, 1]`.
Training and validation come from disjoint portions of the original training set.


In [ ]:
preprocessing = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,)),
])
train_ds, val_ds, test_ds, class_names = cu.fashion_data(
    train_transform=preprocessing, eval_transform=preprocessing,
    train_limit=TRAIN_N, val_limit=VAL_N, test_limit=TEST_N, seed=SEED,
)
train_loader = cu.loader(train_ds, BATCH_SIZE, shuffle=True, seed=SEED)
val_loader = cu.loader(val_ds, BATCH_SIZE)
test_loader = cu.loader(test_ds, BATCH_SIZE)
cu.show_images(train_ds, class_names, n=8, mean=(0.5,), std=(0.5,))


## 3. Apply a hand-designed edge filter

A 3 x 3 filter slides over the image, multiplying and summing nearby pixels.
This particular filter responds to horizontal changes in intensity, often highlighting
vertical edges. Its weights are chosen by us; a CNN learns its own filters from data.

`F.conv2d` expects a batch and channel dimensions. Strictly, PyTorch implements
cross-correlation rather than flipping the kernel as mathematical convolution does.
This naming distinction does not prevent the filters from being learned.


In [ ]:
example, example_label = train_ds[0]
image_01 = (example * 0.5 + 0.5).clamp(0, 1)
edge_kernel = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
edge_response = F.conv2d(image_01.unsqueeze(0),
                         edge_kernel.view(1, 1, 3, 3), padding=1)
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(image_01[0], cmap="gray")
axes[0].set_title(class_names[example_label])
axes[1].imshow(edge_kernel, cmap="coolwarm")
axes[1].set_title("Hand-designed 3 x 3 filter")
axes[2].imshow(edge_response[0, 0], cmap="coolwarm")
axes[2].set_title("Filter response")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Predict convolution shapes and parameter counts

For kernel size `K`, padding `P`, stride `S`, and dilation 1, one spatial dimension becomes
`floor((input_size + 2*P - K) / S) + 1`. With size 28, `K=3`, `P=1`, and `S=1`, it stays 28.
`Conv2d(1, 16, 3, padding=1)` learns 16 filters, so the output has 16 channels.

Parameters = `out_channels * (in_channels * K * K + 1)` when bias is enabled.
The extra one is one bias per output channel. Image height and width do not change this count.


In [ ]:
sample_conv = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
sample_input = torch.zeros(2, 1, 28, 28)
print("Input:", tuple(sample_input.shape))
print("Output:", tuple(sample_conv(sample_input).shape))
print("Weight tensor:", tuple(sample_conv.weight.shape))
print("Expected parameters:", 16 * (1 * 3 * 3 + 1))
print("Actual parameters:", sum(p.numel() for p in sample_conv.parameters()))


## 5. Pool nearby responses

Max pooling with a 2 x 2 window and stride 2 keeps the largest value from each region.
Here 28 x 28 becomes 14 x 14. Pooling has no learned weights. It reduces spatial detail
and computation, but can also discard information useful for a task.

**Ask the audience:** calculate the four pooled values from this 4 x 4 example before running it.


In [ ]:
small_grid = torch.tensor([[1., 2., 3., 4.],
                           [5., 6., 7., 8.],
                           [9., 10., 11., 12.],
                           [13., 14., 15., 16.]])
pooled = F.max_pool2d(small_grid[None, None], kernel_size=2, stride=2)
print("Before:\n", small_grid)
print("After:\n", pooled[0, 0])


## 6. Define a complete, compact CNN

The first block learns 16 feature maps, then pools to 14 x 14. The second learns 32 maps,
then pools to 7 x 7. Flattening produces `32 * 7 * 7 = 1,568` features for a small classifier.
ReLU supplies nonlinear behavior; dropout regularizes the classifier during training.
The ten final outputs are logits for cross-entropy, so there is no final softmax.


In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cu.seed_everything(SEED)
model = FashionCNN().to(DEVICE)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))


## 7. Trace every shape before training

Shape errors are easier to fix with one batch than after starting a long run.
Here we pass a sample through each layer and print the result.
Switching `28` to `32` without adjusting the classifier would change the flattened width;
later architectures use adaptive pooling to handle that more flexibly.


In [ ]:
model.eval()
with torch.no_grad():
    activation = torch.zeros(2, 1, 28, 28, device=DEVICE)
    print("Input:", tuple(activation.shape))
    for layer in list(model.features) + list(model.classifier):
        activation = layer(activation)
        print(f"{layer.__class__.__name__:14s} -> {tuple(activation.shape)}")
assert activation.shape == (2, 10)


## 8. Train the CNN

The reusable `cu.fit` function performs the loops already written in lessons 02 and 03:
train mode, clear gradients, forward, cross-entropy, backward, optimizer step, then validation
in evaluation mode without gradients. It returns the history and restores the lowest
validation-loss checkpoint. You can inspect its implementation in `course_utils.py`.

The model architecture changes; the core training process stays the same.


In [ ]:
history = cu.fit(model, train_loader, val_loader, epochs=EPOCHS,
                 device=DEVICE, lr=1e-3, weight_decay=1e-4)
cu.plot_history(history)
print("Selected validation metrics:", cu.evaluate(model, val_loader, DEVICE))


## 9. Inspect learned first-layer filters

These filters were optimized for classification rather than set by hand.
Each is only 3 x 3, so do not expect a detailed picture of a shoe or coat.
Early filters can respond to local intensity patterns; their role also depends on later layers.
Color uses one shared scale so filter magnitudes can be compared.


In [ ]:
kernels = model.features[0].weight.detach().cpu()[:, 0]
limit = max(kernels.abs().max().item(), 1e-8)
fig, axes = plt.subplots(2, 8, figsize=(13, 4))
for index, ax in enumerate(axes.flat):
    ax.imshow(kernels[index], cmap="coolwarm", vmin=-limit, vmax=limit)
    ax.set_title(f"Filter {index}", fontsize=9)
    ax.axis("off")
fig.suptitle("Learned first-layer filters (shared color scale)")
plt.tight_layout()
plt.show()


## 10. Inspect feature maps for one validation image

A filter is a small learned weight array; a **feature map** is its response across an input.
The maps below are after the first ReLU, before pooling. Different channels can respond
differently to the same clothing item.

**Discussion:** which maps have strong responses around the outline? Which are mostly quiet?
A bright location shows an activation, not proof that this pixel caused the final prediction.


In [ ]:
validation_image, validation_label = val_ds[0]
model.eval()
with torch.no_grad():
    input_batch = validation_image.unsqueeze(0).to(DEVICE)
    feature_maps = model.features[1](model.features[0](input_batch))[0].cpu()
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
axes.flat[0].imshow((validation_image[0] * 0.5 + 0.5).clamp(0, 1), cmap="gray")
axes.flat[0].set_title(f"Input: {class_names[validation_label]}", fontsize=9)
maximum = max(feature_maps.max().item(), 1e-8)
for index in range(16):
    axes.flat[index + 1].imshow(feature_maps[index], cmap="magma", vmin=0, vmax=maximum)
    axes.flat[index + 1].set_title(f"Map {index}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 11. Compare model size without confusing it with quality

A CNN can exploit local structure and shared filters. It can still have many parameters,
especially in flattened fully connected layers. Parameter count alone cannot establish
accuracy, inference speed, or memory use; activations and operations also matter.
This table compares architectures, not trained performance from different experiments.


In [ ]:
linear_count = (28 * 28 + 1) * 10
first_conv_count = sum(p.numel() for p in model.features[0].parameters())
feature_count = sum(p.numel() for p in model.features.parameters())
classifier_count = sum(p.numel() for p in model.classifier.parameters())
print(f"Linear classifier:  {linear_count:,} parameters")
print(f"First convolution: {first_conv_count:,} parameters")
print(f"CNN feature stack: {feature_count:,} parameters")
print(f"CNN classifier:    {classifier_count:,} parameters")
print(f"Complete CNN:      {feature_count + classifier_count:,} parameters")


## 12. Make the final held-out assessment

Use the validation-selected checkpoint and deterministic test preprocessing.
Do not compare this small-run score directly against published full-dataset benchmarks.
The following lesson will look beyond accuracy and examine class-specific errors.


In [ ]:
test_logits, test_labels = cu.predict(model, test_loader, DEVICE)
test_predictions = test_logits.argmax(1)
print(f"Test accuracy: {(test_predictions == test_labels).float().mean().item():.1%}")
print(f"Test loss: {F.cross_entropy(test_logits, test_labels).item():.4f}")
print("Test examples:", len(test_labels))

fig, axes = plt.subplots(2, 4, figsize=(11, 5))
for index, ax in enumerate(axes.flat):
    image, actual = test_ds[index]
    guess = test_predictions[index].item()
    ax.imshow((image[0] * 0.5 + 0.5).clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"True: {class_names[actual]}\nPred: {class_names[guess]}",
                 color="green" if actual == guess else "crimson", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 13. Save a self-described checkpoint

Save CPU tensors so the checkpoint can later be loaded on a different device.
We retain architecture and preprocessing information alongside the selected weights.
Rerunning this cell replaces this lesson's named artifact; other lessons train independently.


In [ ]:
checkpoint_path = cu.ARTIFACTS / "04_fashion_cnn.pt"
torch.save({
    "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    "architecture": "FashionCNN", "class_names": list(class_names),
    "image_size": [28, 28], "mean": [0.5], "std": [0.5],
    "seed": SEED, "smoke": SMOKE, "train_examples": len(train_ds),
}, checkpoint_path)
print(checkpoint_path)


## 14. Exercises and demonstration prompts

1. Predict the shape after a 5 x 5 convolution with no padding on a 28 x 28 image.
2. Count the parameters of `Conv2d(16, 32, 3, padding=1)` without inspecting the model.
3. Replace max pooling with average pooling and compare validation performance using
   fresh models, the same seed, and the same training budget.
4. Change the first layer to accept RGB input. What else must change in the input pipeline?

<details><summary>Solution hints</summary>

The convolution produces 24 x 24 spatial dimensions. The second convolution has
`32 * (16 * 3 * 3 + 1) = 4,640` parameters. Average pooling summarizes the mean rather
than the maximum; whether it helps is empirical. RGB inputs need three channels,
appropriate normalization, and data whose resolution matches the classifier.

```python
print(nn.Conv2d(1, 16, 5)(torch.zeros(2, 1, 28, 28)).shape)
layer = nn.Conv2d(16, 32, 3, padding=1)
print(sum(p.numel() for p in layer.parameters()))
# Replace both nn.MaxPool2d(2) layers with nn.AvgPool2d(2), then retrain.
```

</details>

**Presenter checkpoint:** trace the shape from input to logits without running the model.
Explain the difference between a filter and a feature map in your own words.

**Limitations:** these visualizations are exploratory; they are not causal explanations.
The architecture assumes 28 x 28 grayscale inputs and has not been validated for production.

**References:** [Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html),
[MaxPool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html),
[PyTorch image-classification tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html).
